In [ ]:
%matplotlib inline
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# 6.2 run: v1+v2+v3hard stratified, 200 steps, Qwen3.5-9B GRPO
# Steps 1–39: pre-OOM (from assets S3, no policy_loss/kl/entropy/grad_norm)
# Steps 41–160: post-OOM-fix restart from checkpoint 40
METRICS_DIR = "metrics/"
all_steps = json.load(open(METRICS_DIR + "training_metrics_6.2_all.json"))
all_steps.sort(key=lambda x: x["step"])
print(f"Total steps: {len(all_steps)}, range: {all_steps[0]['step']}–{all_steps[-1]['step']}")
print("Note: policy_loss/kl/entropy/grad_norm only available from step 41 onward")

steps           = [m["step"]                         for m in all_steps]
raw_reward      = [m.get("avg_raw_reward",    None)  for m in all_steps]
std_reward      = [m.get("std_reward",        None)  for m in all_steps]
pass2           = [m.get("avg_pass_at_2",     None)  for m in all_steps]
entropy         = [m.get("policy_entropy",    None)  for m in all_steps]
grad_norm       = [m.get("grad_norm",         None)  for m in all_steps]
policy_kl       = [m.get("policy_kl",         None)  for m in all_steps]
response_length = [m.get("response_length",   None)  for m in all_steps]

def rolling(y, w=5):
    out = []
    for i in range(len(y)):
        window = [v for v in y[max(0, i-w+1):i+1] if v is not None]
        out.append(np.mean(window) if window else None)
    return out


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
fig.suptitle("Phase 2 Run 6.2 — Qwen3.5-9B GRPO (v1+v2+v3hard stratified, steps 1–160)\n"
             "Dashed line at step 40 = OOM crash + restart", fontsize=12, fontweight="bold")

OOM_STEP = 40  # vertical marker: OOM crash + restart point

def plot_panel(ax, y, title, color, std=None):
    r = rolling(y)
    ax.plot(steps, y, color=color, alpha=0.3, linewidth=1, label="per step")
    ax.plot(steps, r, color=color, linewidth=2, label="rolling avg (5)")
    if std is not None:
        r_std = rolling(std)
        r_arr = np.array([v if v is not None else np.nan for v in r])
        s_arr = np.array([v if v is not None else np.nan for v in r_std])
        ax.fill_between(steps, r_arr - s_arr, r_arr + s_arr, color=color, alpha=0.12, label="± std (rolling)")
    ax.axvline(OOM_STEP, color="red", linestyle="--", linewidth=1, alpha=0.6, label="OOM/restart")
    ax.set_title(title)
    ax.set_xlabel("step")
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.grid(True, alpha=0.2)
    ax.legend(fontsize=7)

plot_panel(axes[0,0], raw_reward,      "Avg Raw Reward",   "#2196F3", std=std_reward)
plot_panel(axes[0,1], pass2,           "Avg Pass@2",       "#4CAF50")
plot_panel(axes[0,2], std_reward,      "Std Reward",       "#F44336")
plot_panel(axes[1,0], entropy,         "Policy Entropy",   "#FF9800")
plot_panel(axes[1,1], grad_norm,       "Grad Norm",        "#9C27B0")
plot_panel(axes[1,2], policy_kl,       "Policy KL",        "#00BCD4")
plot_panel(axes[2,0], response_length, "Response Length",  "#795548")
axes[2,1].axis("off")
axes[2,2].axis("off")

plt.tight_layout()
plt.savefig("metrics/training_metrics_plot_6.2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: metrics/training_metrics_plot_6.2.png")


In [ ]:
# Collapse Detection
# entropy collapse threshold is lower for 6.2 — entropy is already declining toward 0.06

ENTROPY_COLLAPSE_THRESHOLD = 0.03   # 6.2 entropy reached 0.064 by steps 101-150; watch for further drop
STD_COLLAPSE_THRESHOLD     = 0.05
KL_DIVERGE_THRESHOLD       = 0.5
GRAD_SPIKE_THRESHOLD       = 5.0
SUSTAINED_ZERO_WINDOW      = 3

flags = []

for i, m in enumerate(all_steps):
    s   = m["step"]
    ent = m.get("policy_entropy", 1.0)
    std = m.get("std_reward", 1.0)
    kl  = m.get("policy_kl", 0.0)
    gn  = m.get("grad_norm", 0.0)
    if ent is not None and ent < ENTROPY_COLLAPSE_THRESHOLD:
        flags.append((s, "ENTROPY_COLLAPSE", f"entropy={ent:.4f}"))
    if kl > KL_DIVERGE_THRESHOLD:
        flags.append((s, "KL_DIVERGENCE",    f"kl={kl:.4f}"))
    if gn > GRAD_SPIKE_THRESHOLD:
        flags.append((s, "GRAD_SPIKE",       f"grad_norm={gn:.4f}"))

for i in range(len(all_steps) - SUSTAINED_ZERO_WINDOW + 1):
    window = all_steps[i:i+SUSTAINED_ZERO_WINDOW]
    if all(m.get("std_reward", 1.0) < STD_COLLAPSE_THRESHOLD for m in window):
        step_range = f"{window[0]['step']}–{window[-1]['step']}"
        flags.append((window[0]["step"], "SUSTAINED_ZERO_REWARD", f"steps {step_range}"))

print("=== Collapse Detection — 6.2 Run ===")
print(f"Thresholds: entropy<{ENTROPY_COLLAPSE_THRESHOLD}, std_reward<{STD_COLLAPSE_THRESHOLD} "
      f"for {SUSTAINED_ZERO_WINDOW} consecutive steps, kl>{KL_DIVERGE_THRESHOLD}, grad_norm>{GRAD_SPIKE_THRESHOLD}")
print(f"Note: entropy in 6.2 averages 0.064 in steps 101–150 — well below 6.1's stable 0.30.")
print()

if flags:
    print(f"{'step':>5}  {'type':<25}  detail")
    for step, ftype, detail in sorted(flags):
        print(f"{step:>5}  {ftype:<25}  {detail}")
else:
    print("No collapse signals detected.")

print()
low_std_steps = [(m["step"], m.get("std_reward", 0)) for m in all_steps if m.get("std_reward", 1.0) < STD_COLLAPSE_THRESHOLD]
print(f"=== Steps with std_reward < {STD_COLLAPSE_THRESHOLD} ({len(low_std_steps)} total) ===")
if low_std_steps:
    print(f"{'step':>5}  {'std_reward':>10}")
    for s, v in low_std_steps:
        print(f"{s:>5}  {v:>10.4f}")
else:
    print("None")

print()
print("=== Per-step Summary (last 10) ===")
print(f"{'step':>5}  {'entropy':>8}  {'std_rwd':>8}  {'kl':>7}  {'grad_norm':>9}  status")
for m in all_steps[-10:]:
    ent = m.get("policy_entropy")
    std = m.get("std_reward", 0)
    kl  = m.get("policy_kl", 0)
    gn  = m.get("grad_norm", 0)
    if ent is None:
        status = "  N/A"
        print(f"{m['step']:>5}  {'N/A':>8}  {std:>8.4f}  {kl:>7.3f}  {gn:>9.3f}  {status}")
    else:
        ok = ent > ENTROPY_COLLAPSE_THRESHOLD and kl < KL_DIVERGE_THRESHOLD and gn < GRAD_SPIKE_THRESHOLD
        status = "✓ ok" if ok else "✗ FLAG"
        print(f"{m['step']:>5}  {ent:>8.3f}  {std:>8.4f}  {kl:>7.3f}  {gn:>9.3f}  {status}")


In [ ]:
from IPython.display import Markdown, display

METRICS = [
    ("avg_pass_at_2",          "avg_pass@2",     "{:.3f}"),
    ("std_reward",             "std_reward",     "{:.3f}"),
    ("policy_kl",              "policy_kl",      "{:.3f}"),
    ("policy_entropy",         "policy_entropy", "{:.3f}"),
    ("grad_norm",              "grad_norm",      "{:.3f}"),
    ("response_length",        "response_length","{:,.0f}"),
    ("num_error_trajectories", "num_errors",     "{:.2f}"),
    ("policy_loss",            "policy_loss",    "{:.4f}"),
]

PHASES = [(1, 50), (51, 100), (101, 150), (151, None)]

def phase_avg(key, lo, hi):
    vals = [m[key] for m in all_steps
            if lo <= m["step"] and (hi is None or m["step"] <= hi)
            and m.get(key) is not None]
    return np.mean(vals) if vals else float("nan")

last_step = all_steps[-1]["step"]
headers = ["Steps 1–50", "Steps 51–100", "Steps 101–150", f"Steps 151–{last_step}"]

rows = []
for key, label, fmt in METRICS:
    cells = []
    for lo, hi in PHASES:
        avg = phase_avg(key, lo, hi)
        cells.append(fmt.format(avg) if not np.isnan(avg) else "—")
    rows.append(f"| {label} | {' | '.join(cells)} |")

table = "\n".join([
    f"### 6.2 Phase Averages (steps 1–{last_step})",
    "",
    "| Metric | " + " | ".join(headers) + " |",
    "|--------|" + "|".join(["--------"] * len(headers)) + "|",
] + rows)

display(Markdown(table))


# Metric Insights — 6.2 Run

**Pass@2** — did the model solve the task in at least 1 of 2 tries.
6.1 climbed from 0.1 → 0.5. 6.2 flat at 0.4 the whole time.
*6.1 was genuinely getting better. 6.2 never improved from where it started.*

**Entropy** — how many different approaches the model tries. Low = always does the same thing.
6.1 rose from 0.18 → 0.30. 6.2 fell from 0.16 → 0.04.
*6.1 kept exploring new strategies. 6.2 narrowed to a small set of commands and stopped.*

**Policy KL** — how far the model has moved from where it began training.
6.1 rose from 0.15 → 0.25. 6.2 fell from 0.18 → 0.12.
*6.1's weights changed significantly — it actually learned something. 6.2 barely moved.*

**Grad norm** — how big each weight update is.
6.1 was mostly flat with one early spike. 6.2 is slowly rising with a sharp spike around step 160.
*6.2's updates are getting larger as entropy collapses — the model is doubling down on its narrow strategy, not stabilizing.*

**Std reward** — mix of passes and fails per step. Zero means all same outcome, no learning signal.
6.1 was unstable early (many zero dips) then stabilized. 6.2 stayed consistently healthy around 0.43–0.50.
*6.2 had a cleaner signal throughout, which ironically made it converge too fast.*

**Response length** — how much the model wrote across all tasks per step.
6.1 grew from ~2.5k → ~7k+ and was heading toward OOM. 6.2 stayed mostly under 5k with occasional spikes.
*The max_input_tokens cap in 6.2 worked — no OOM risk. 6.1 was on a bad trajectory.*

**avg_pass@2** — fraction of tasks the model solved in at least 1 of its 2 attempts per step.
Started strong at 0.416 in steps 1–50 (vs 6.1's 0.235 — but 6.1 had broken Docker in that window). Both runs converge to ~0.43 by steps 51–100. 6.2 holds at 0.383 in steps 101–150. No clear upward trend — pass@2 is noisy step-to-step because only 8 tasks are sampled per step.

**std_reward** — how mixed the results are each step. Zero means all tasks passed or all failed — no learning signal.
Stayed above 0.43 throughout all phases. The model always had a mix of passes and failures, so the GRPO gradient was never starved. The occasional zero-std steps (steps 60, 135) were isolated single-step events, not sustained collapse.

**policy_entropy** — how varied the model's outputs are. High = tries many different approaches; low = always does the same thing.
This is the main warning signal in 6.2. Entropy dropped steadily: 0.149 (steps 1–50) → 0.120 (51–100) → 0.064 (101–150). Compare to 6.1 which stayed flat around 0.29–0.30. The model is converging to a narrow set of strategies much faster than 6.1 did. If entropy keeps falling past step 160, the model risks locking into repetitive behavior before fully learning the task space.

**grad_norm** — size of each weight update. Very high values mean unstable training.
Rising slowly: 0.47 → 0.53 → 0.61. Still well below the danger threshold of 5.0, but the upward trend alongside falling entropy is consistent with the model making increasingly confident (but narrower) updates.

**policy_kl** — how far the current policy has drifted from the reference model.
Slightly declining: 0.169 → 0.167 → 0.135. Well below the 0.5 warning threshold. 6.1 showed the opposite trend (rising KL toward 0.22) — 6.2's model is staying closer to its initialization even as it trains, which may partly explain the entropy drop.

**response_length** — total tokens generated across all tasks in a step.
Stayed controlled at 3,577 / 3,206 / 4,517 across the three phases — much better than 6.1's growth to ~7k. This is the direct result of capping `max_input_tokens=16384`. No OOM risk.

**num_errors** — Docker/infrastructure failures per step (out of 16 trajectories).
Flat at ~0.24 across all phases — about 1 Docker error every 4 steps. Low and stable; the Docker backend fix is working.

**policy_loss** — weight of updates each step. Not available for steps 1–39 (pre-restart).
Near zero throughout (0.015 → 0.005 → 0.012). No signs of instability.

---
**Summary:** 6.2 is running cleanly with no infrastructure issues and pass@2 holding at ~0.38–0.43. The main thing to watch going into steps 160–200 is the entropy decline — if it drops below 0.03 the model may stop exploring and pass@2 could plateau or fall.
